# Hackathon ML 2026 : notebook d'équipe

Cible : `intensity` (easy / moderate / hard). Métrique : macro-F1.
Toute la validation passe par `src/validation.py` (5 folds stratifiés, graine 42).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

from src.validation import load_data, evaluate, plot_confusion, make_submission, SEED
from src.features import UserStats

pd.set_option("display.max_columns", 60)

## 1. Chargement

In [ ]:
train, test, sample = load_data()
ID_COL, TARGET = "workout_id", "intensity"
print(train.shape, test.shape)
train.head()

## 2. Exploration rapide

In [ ]:
train.info()
print("\nValeurs manquantes :")
print(train.isna().mean().sort_values(ascending=False).head(15))

In [ ]:
print(train[TARGET].value_counts(normalize=True).round(3))
train.describe().T

In [ ]:
# Colonnes catégorielles : nombre de modalités et lien avec la cible
cat_cols = [c for c in train.select_dtypes(exclude="number").columns if c != TARGET]
for c in cat_cols:
    print(f"{c}: {train[c].nunique()} modalités")
    if train[c].nunique() <= 15:
        display(pd.crosstab(train[c], train[TARGET], normalize="index").round(2))

In [ ]:
# Variables numériques par classe
num_cols = [c for c in train.select_dtypes(include="number").columns if c != ID_COL]
train.groupby(TARGET)[num_cols].median().T

## 3. Variables

**Idée centrale :** `intensity` est mesurée *pour chaque personne* (environ 1/3 de chaque classe pour chaque utilisateur). Une vitesse de 12 km/h n'a donc pas
le même sens pour deux personnes : on compare chaque séance **aux habitudes du même utilisateur dans le même sport**.

`UserStats` (`src/features.py`) calcule, pour la vitesse, la durée, la distance, la vitesse max, le dénivelé, etc. :
le **ratio à sa médiane**, l'**écart en écarts-types** et le **rang** de la séance parmi les siennes.
C'est un transformer scikit-learn placé *dans* le pipeline : à chaque fold, il n'apprend les habitudes que sur le train du fold, donc pas de fuite de la cible.

In [ ]:
ID_COL, TARGET = "workout_id", "intensity"
X = train.drop(columns=[ID_COL, TARGET])      # user_id et sport restent : UserStats en a besoin pour les habitudes
y = train[TARGET]
X_test = test.drop(columns=[ID_COL])
print(X.shape, X_test.shape)
print("UserStats ->", UserStats().fit(X).transform(X).shape[1], "variables")

## 4. Comparaison de 3 familles de modèles (exigée pour le LOG.md)

In [ ]:
def pipe(model, scale=False):
    steps = [UserStats(), SimpleImputer(strategy="median")]
    if scale:
        steps.append(StandardScaler())
    return make_pipeline(*steps, model)

models = {
    "logreg": pipe(LogisticRegression(max_iter=3000), scale=True),
    "knn": pipe(KNeighborsClassifier(n_neighbors=25), scale=True),
    "random_forest": pipe(RandomForestClassifier(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=SEED)),
    "hist_gb": make_pipeline(UserStats(), HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=SEED)),
}

results = {}
for name, model in models.items():
    print(f"--- {name}")
    results[name] = evaluate(model, X, y)

pd.DataFrame({k: (v["mean"], v["std"]) for k, v in results.items()}, index=["macro-F1", "std"]).T.sort_values("macro-F1", ascending=False)

## 5. Analyse d'erreur (Gate 2)

In [ ]:
best = max(results, key=lambda k: results[k]["mean"])
print("Meilleur modèle :", best)
plot_confusion(y, results[best]["oof_pred"], title=f"{best} : matrice de confusion hors fold")

In [ ]:
# Exemples mal classés à regarder à la main
errors = train.assign(pred=results[best]["oof_pred"]).query(f"pred != {TARGET}")
print(len(errors), "erreurs")
errors.sample(10, random_state=SEED)

## 6. Soumission

Avant chaque envoi : ajouter une ligne dans `LOG.md`.

In [ ]:
SUB_NUM = 1
sub = make_submission(models[best], X, y, X_test, test[ID_COL], f"submissions/sub_{SUB_NUM:02d}_{best}.csv")
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)
assert (sub[ID_COL].values == sample[ID_COL].values).all()